# Notebook 17 – Model Service Design

**Goal:** move from *notebook code* toward *reusable AI engineering code*.

We build a small, modular model-inference service with these parts:

| Part | Responsibility |
|---|---|
| `ModelConfig` | All settings in one place |
| `get_logger` | Consistent logging |
| `ModelServiceError` family | Clear, specific errors |
| `Preprocessor` | Clean and validate input |
| `ModelLoader` | Load the model once, reuse it |
| `PostProcessor` | Turn raw model output into a clean result |
| `InferenceService` | Connect all parts together |

> **Offline demo note:** the outputs below use a tiny built-in `stub` backend so the notebook runs anywhere with no downloads.
> To use a real Hugging Face model, install `transformers` and set `backend="huggingface"` in `ModelConfig`. Nothing else in the code changes - that is the point of good design.

## 1. Notebook Code vs Service Code

**Concept:** Notebook code is quick and works for one experiment. Service code is organised so it can be reused, tested, and trusted by other people.

**Why it's needed:** A function like the one below has hard-coded words, no input checks, no logging, and no clear errors. It breaks the moment real input arrives.

In [1]:
# Notebook-style code: works, but is hard to reuse, test, or debug
def quick_sentiment(text):
    words = text.lower().split()
    good = sum(w in ["great", "good", "love"] for w in words)
    bad = sum(w in ["bad", "slow", "hate"] for w in words)
    return "POSITIVE" if good >= bad else "NEGATIVE"

print(quick_sentiment("The delivery was great"))

try:
    print(quick_sentiment(None))
except Exception as exc:
    print("Crash:", type(exc).__name__, "-", exc)

POSITIVE
Crash: AttributeError - 'NoneType' object has no attribute 'lower'


**What this example tells us**
- It works for good input but crashes with an unclear error for bad input.
- Settings, cleaning, model logic, and output are all mixed in one place.

**AI/ML Example:** A notebook cell that loads a model, cleans text, predicts, and prints - all in one block - cannot be reused in an API or a batch job.

**Business Example:** A support-ticket classifier that crashes on an empty ticket will stop the whole queue.

**AI/ML Engineer Use Case:** Spot notebook patterns that must become modules before code goes to production.

## 2. ModelConfig – Configuration

**Concept:** A config class stores every setting (model name, batch size, threshold) in one place and validates it.

**Why it's needed:** Hard-coded values scattered across code are easy to forget and change. One config object makes behaviour clear, safe, and easy to change.

In [2]:
from dataclasses import dataclass, replace

@dataclass(frozen=True)
class ModelConfig:
    """All settings for the model service, validated on creation."""
    model_name: str = "distilbert-base-uncased-finetuned-sst-2-english"
    task: str = "sentiment-analysis"
    backend: str = "stub"            # "stub" (offline demo) or "huggingface"
    max_length: int = 128            # max words kept per input
    batch_size: int = 8
    confidence_threshold: float = 0.60
    device: int = -1                 # -1 = CPU

    def __post_init__(self) -> None:
        if self.backend not in {"stub", "huggingface"}:
            raise ValueError(f"backend must be 'stub' or 'huggingface', got '{self.backend}'")
        if self.max_length <= 0 or self.batch_size <= 0:
            raise ValueError("max_length and batch_size must be positive")
        if not 0.0 <= self.confidence_threshold <= 1.0:
            raise ValueError("confidence_threshold must be between 0 and 1")

config = ModelConfig()
print(config)

try:
    ModelConfig(confidence_threshold=1.5)
except ValueError as exc:
    print("Invalid config:", exc)

strict = replace(config, confidence_threshold=0.90)
print("Original threshold:", config.confidence_threshold)
print("New threshold     :", strict.confidence_threshold)

ModelConfig(model_name='distilbert-base-uncased-finetuned-sst-2-english', task='sentiment-analysis', backend='stub', max_length=128, batch_size=8, confidence_threshold=0.6, device=-1)
Invalid config: confidence_threshold must be between 0 and 1
Original threshold: 0.6
New threshold     : 0.9

**What this example tells us**
- Settings live in one object with type hints and defaults.
- Bad values fail early, with a clear message.
- `frozen=True` means the config cannot be changed by accident; `replace()` makes a safe modified copy.

**AI/ML Example:** Switching from a small model to a larger one by changing only `model_name`.

**Business Example:** Raising the confidence threshold for a refund-approval model without touching any code logic.

**AI/ML Engineer Use Case:** Load the same config from a YAML/JSON file or environment variables in different environments (dev, test, production).

## 3. Logging

**Concept:** Logging records what the service is doing, with levels such as DEBUG, INFO, WARNING, and ERROR.

**Why it's needed:** `print()` cannot be filtered, routed to files, or turned off. Logs help you find problems after something has already gone wrong.

In [3]:
import logging
import sys

def get_logger(name: str, level: int = logging.INFO) -> logging.Logger:
    """Return a logger with one clean console handler (no duplicate lines)."""
    logger = logging.getLogger(name)
    if not logger.handlers:
        handler = logging.StreamHandler(sys.stdout)
        handler.setFormatter(logging.Formatter("%(levelname)s | %(name)s | %(message)s"))
        logger.addHandler(handler)
    logger.setLevel(level)
    logger.propagate = False
    return logger

logger = get_logger("model_service")
logger.info("Service starting")
logger.warning("Low confidence prediction")
logger.debug("This debug line is hidden at INFO level")

INFO | model_service | Service starting
WARNING | model_service | Low confidence prediction


**What this example tells us**
- Each message has a level and a source name.
- The DEBUG line is hidden because the logger level is INFO.
- The helper avoids adding the same handler twice (a common notebook problem when cells are re-run).

**AI/ML Example:** Log model name, batch size, and prediction time for every inference run.

**Business Example:** When a customer reports a wrong prediction, logs show what the system did at that time.

**AI/ML Engineer Use Case:** Add `%(asctime)s` to the format and send logs to a file or monitoring tool in production.

## 4. Custom Exceptions – Error Handling

**Concept:** Custom exception classes describe *what kind* of failure happened: bad input, model loading failure, or inference failure.

**Why it's needed:** Generic errors like `AttributeError` do not tell the caller what to do. Specific errors let callers react correctly (reject the request, retry, or alert).

In [4]:
class ModelServiceError(Exception):
    """Base class for all errors raised by the model service."""

class ModelLoadError(ModelServiceError):
    """The model could not be loaded."""

class InvalidInputError(ModelServiceError):
    """The input text is missing or not valid."""

class InferenceError(ModelServiceError):
    """The model failed while making predictions."""

def risky_call(value: str) -> str:
    """Tiny example that raises a clear, specific error."""
    if not value:
        raise InvalidInputError("Input text is empty")
    return value

try:
    risky_call("")
except ModelServiceError as exc:          # one base class catches them all
    print(f"Caught {type(exc).__name__}: {exc}")

Caught InvalidInputError: Input text is empty

**What this example tells us**
- Each error has a clear name and message.
- Catching the base class `ModelServiceError` handles every service error in one place.

**AI/ML Example:** Return HTTP 400 for `InvalidInputError` and HTTP 500 for `InferenceError`.

**Business Example:** A bad customer input is rejected politely, while a real system failure triggers an alert to the team.

**AI/ML Engineer Use Case:** Build predictable APIs where every failure has a known type and response.

## 5. Preprocessor

**Concept:** The `Preprocessor` has one job: validate and clean input text before it reaches the model.

**Why it's needed:** Models give poor or unpredictable results on messy input. Keeping cleaning in one class means every caller gets the same behaviour.

In [5]:
from typing import Any, List, Optional, Sequence

class Preprocessor:
    """Validates and cleans raw text before inference."""

    def __init__(self, config: ModelConfig, logger: Optional[logging.Logger] = None) -> None:
        self.config = config
        self.logger = logger or get_logger("model_service.preprocessor")

    def clean(self, text: Any) -> str:
        """Return cleaned text or raise InvalidInputError."""
        if not isinstance(text, str):
            raise InvalidInputError(f"Expected str, got {type(text).__name__}")
        cleaned = " ".join(text.split())          # trim and collapse whitespace
        if not cleaned:
            raise InvalidInputError("Input text is empty")
        words = cleaned.split(" ")
        if len(words) > self.config.max_length:
            self.logger.warning("Input truncated from %d to %d words", len(words), self.config.max_length)
            cleaned = " ".join(words[: self.config.max_length])
        return cleaned

    def clean_batch(self, texts: Sequence[Any]) -> List[str]:
        """Clean a list of texts; fails clearly if the list is empty."""
        if not texts:
            raise InvalidInputError("No texts were provided")
        return [self.clean(t) for t in texts]

pre = Preprocessor(config)
print(repr(pre.clean("  The   delivery was GREAT!  \n")))

short_pre = Preprocessor(replace(config, max_length=5))
print(repr(short_pre.clean("one two three four five six seven")))

for bad in [None, "   ", 123]:
    try:
        pre.clean(bad)
    except InvalidInputError as exc:
        print(f"{bad!r:>6} -> InvalidInputError: {exc}")

'The delivery was GREAT!'
WARNING | model_service.preprocessor | Input truncated from 7 to 5 words
'one two three four five'
  None -> InvalidInputError: Expected str, got NoneType
 '   ' -> InvalidInputError: Input text is empty
   123 -> InvalidInputError: Expected str, got int

**What this example tells us**
- Extra spaces and line breaks are cleaned.
- Long input is truncated, with a warning logged.
- Empty, `None`, and non-text input raise a clear `InvalidInputError`.

**AI/ML Example:** Cleaning and length-limiting text before tokenization so the model never receives an unexpected input.

**Business Example:** A chatbot receives blank messages and pasted emails; the preprocessor handles both safely.

**AI/ML Engineer Use Case:** Add new cleaning rules (remove HTML, normalise Unicode) in one place without touching model code.

## 6. PostProcessor

**Concept:** The `PostProcessor` converts raw model output into a clean, predictable result object.

**Why it's needed:** Raw model output (a dict with a label and score) changes between models. Callers should always receive the same clean structure.

In [6]:
from dataclasses import dataclass, asdict
from typing import Any, Dict

@dataclass(frozen=True)
class Prediction:
    """Clean result returned by the service."""
    text: str
    label: str
    score: float
    is_confident: bool

    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)

class PostProcessor:
    """Turns raw model output into Prediction objects."""

    def __init__(self, config: ModelConfig, logger: Optional[logging.Logger] = None) -> None:
        self.config = config
        self.logger = logger or get_logger("model_service.postprocessor")

    def format(self, text: str, raw: Dict[str, Any]) -> Prediction:
        """Validate one raw result and apply the confidence threshold."""
        try:
            label = str(raw["label"]).upper()
            score = round(float(raw["score"]), 4)
        except (KeyError, TypeError, ValueError) as exc:
            raise InferenceError(f"Unexpected model output: {raw!r}") from exc
        is_confident = score >= self.config.confidence_threshold
        if not is_confident:
            self.logger.warning("Low confidence (%.2f) for text: %s", score, text)
        return Prediction(text=text, label=label, score=score, is_confident=is_confident)

post = PostProcessor(config)
print(post.format("Great service", {"label": "positive", "score": 0.987654}).to_dict())
print(post.format("It arrived", {"label": "NEGATIVE", "score": 0.52}).to_dict())

try:
    post.format("Broken output", {"unexpected": "format"})
except InferenceError as exc:
    print("InferenceError:", exc)

{'text': 'Great service', 'label': 'POSITIVE', 'score': 0.9877, 'is_confident': True}
WARNING | model_service.postprocessor | Low confidence (0.52) for text: It arrived
{'text': 'It arrived', 'label': 'NEGATIVE', 'score': 0.52, 'is_confident': False}
InferenceError: Unexpected model output: {'unexpected': 'format'}

**What this example tells us**
- Output is always a `Prediction` with the same four fields.
- The confidence threshold is applied in one place.
- Unexpected model output is caught and reported clearly.

**AI/ML Example:** Mapping labels like `LABEL_0` / `LABEL_1` to readable names such as `NEGATIVE` / `POSITIVE`.

**Business Example:** Predictions below the threshold are sent to a human reviewer instead of being trusted automatically.

**AI/ML Engineer Use Case:** Swap the model and keep the same output format, so downstream apps do not break.

## 7. ModelLoader

**Concept:** The `ModelLoader` loads the model once, keeps it in memory, and hands it out when needed (lazy loading + caching).

**Why it's needed:** Loading a model is slow and memory-heavy. Doing it on every request wastes time and resources.

In [7]:
from typing import Callable, Dict, List

# A "backend" is any callable: list of texts -> list of {"label", "score"} dicts
Backend = Callable[[List[str]], List[Dict[str, Any]]]

class StubSentimentBackend:
    """Tiny rule-based stand-in so this notebook runs offline."""
    POSITIVE = {"great", "good", "love", "excellent", "fast"}
    NEGATIVE = {"bad", "slow", "hate", "terrible", "broken"}

    def __call__(self, texts: List[str]) -> List[Dict[str, Any]]:
        results = []
        for text in texts:
            words = [w.strip(".,!?").lower() for w in text.split()]
            pos = sum(w in self.POSITIVE for w in words)
            neg = sum(w in self.NEGATIVE for w in words)
            label = "NEGATIVE" if neg > pos else "POSITIVE"
            score = min(0.99, 0.50 + 0.20 * abs(pos - neg))
            results.append({"label": label, "score": score})
        return results

class ModelLoader:
    """Loads the model once and reuses it."""

    def __init__(self, config: ModelConfig, logger: Optional[logging.Logger] = None,
                 backend_factory: Optional[Callable[[ModelConfig], Backend]] = None) -> None:
        self.config = config
        self.logger = logger or get_logger("model_service.loader")
        self._factory = backend_factory or self._default_factory
        self._model: Optional[Backend] = None

    @property
    def is_loaded(self) -> bool:
        return self._model is not None

    def load(self) -> Backend:
        """Return the cached model, loading it first if needed."""
        if self._model is not None:
            self.logger.info("Model already loaded - reusing")
            return self._model
        self.logger.info("Loading model '%s' (backend=%s)", self.config.model_name, self.config.backend)
        try:
            self._model = self._factory(self.config)
        except Exception as exc:
            self.logger.error("Model loading failed: %s", exc)
            raise ModelLoadError(f"Could not load '{self.config.model_name}': {exc}") from exc
        self.logger.info("Model loaded successfully")
        return self._model

    @staticmethod
    def _default_factory(config: ModelConfig) -> Backend:
        if config.backend == "stub":
            return StubSentimentBackend()
        from transformers import pipeline          # imported only when really needed
        pipe = pipeline(config.task, model=config.model_name, device=config.device)
        return lambda texts: pipe(texts, batch_size=config.batch_size, truncation=True)

loader = ModelLoader(config)
print("Loaded before?", loader.is_loaded)
model = loader.load()
model = loader.load()                               # second call uses the cache
print("Loaded after? ", loader.is_loaded)
print(model(["The delivery was great"]))

def broken_factory(cfg: ModelConfig) -> Backend:
    raise RuntimeError("weights file missing")

try:
    ModelLoader(config, backend_factory=broken_factory).load()
except ModelLoadError as exc:
    print("ModelLoadError:", exc)

Loaded before? False
INFO | model_service.loader | Loading model 'distilbert-base-uncased-finetuned-sst-2-english' (backend=stub)
INFO | model_service.loader | Model loaded successfully
INFO | model_service.loader | Model already loaded - reusing
Loaded after?  True
[{'label': 'POSITIVE', 'score': 0.7}]
INFO | model_service.loader | Loading model 'distilbert-base-uncased-finetuned-sst-2-english' (backend=stub)
ERROR | model_service.loader | Model loading failed: weights file missing
ModelLoadError: Could not load 'distilbert-base-uncased-finetuned-sst-2-english': weights file missing

**What this example tells us**
- The first call loads the model; the second reuses it.
- Real Hugging Face code is imported only when `backend="huggingface"`.
- Any loading failure becomes a clear `ModelLoadError`.
- `backend_factory` lets us inject a fake or broken model - very useful for testing.

**AI/ML Example:** Loading a transformer once at app start-up instead of on every API request.

**Business Example:** Faster response times and lower cloud cost because the model is not reloaded repeatedly.

**AI/ML Engineer Use Case:** Test error handling and the full service without downloading a real model.

## 8. InferenceService

**Concept:** The `InferenceService` is the main class. It does not do the work itself - it *coordinates* the Preprocessor, ModelLoader, and PostProcessor.

**Why it's needed:** Callers need one simple entry point (`predict`, `predict_batch`) and should not know how the internals work.

In [8]:
class InferenceService:
    """Single entry point for model inference."""

    def __init__(self, config: Optional[ModelConfig] = None, *,
                 loader: Optional[ModelLoader] = None,
                 preprocessor: Optional[Preprocessor] = None,
                 postprocessor: Optional[PostProcessor] = None,
                 logger: Optional[logging.Logger] = None) -> None:
        self.config = config or ModelConfig()
        self.logger = logger or get_logger("model_service.inference")
        self.loader = loader or ModelLoader(self.config)
        self.preprocessor = preprocessor or Preprocessor(self.config)
        self.postprocessor = postprocessor or PostProcessor(self.config)

    def predict(self, text: str) -> Prediction:
        """Predict for a single text."""
        return self.predict_batch([text])[0]

    def predict_batch(self, texts: Sequence[str]) -> List[Prediction]:
        """Predict for many texts, processed in chunks of config.batch_size."""
        cleaned = self.preprocessor.clean_batch(texts)
        model = self.loader.load()
        self.logger.info("Running inference on %d text(s)", len(cleaned))
        size = self.config.batch_size
        predictions: List[Prediction] = []
        for start in range(0, len(cleaned), size):
            chunk = cleaned[start:start + size]
            try:
                raw = model(chunk)
            except Exception as exc:
                self.logger.error("Inference failed: %s", exc)
                raise InferenceError(f"Model failed on batch starting at index {start}: {exc}") from exc
            if len(raw) != len(chunk):
                raise InferenceError(f"Model returned {len(raw)} results for {len(chunk)} inputs")
            predictions.extend(self.postprocessor.format(t, r) for t, r in zip(chunk, raw))
        return predictions

    def health(self) -> Dict[str, Any]:
        """Small status report, useful for monitoring."""
        return {"model": self.config.model_name, "backend": self.config.backend,
                "model_loaded": self.loader.is_loaded}

service = InferenceService()
print("Health before:", service.health())
result = service.predict("The delivery was great and support was good")
print(result)
print("Health after: ", service.health())

Health before: {'model': 'distilbert-base-uncased-finetuned-sst-2-english', 'backend': 'stub', 'model_loaded': False}
INFO | model_service.loader | Loading model 'distilbert-base-uncased-finetuned-sst-2-english' (backend=stub)
INFO | model_service.loader | Model loaded successfully
INFO | model_service.inference | Running inference on 1 text(s)
Prediction(text='The delivery was great and support was good', label='POSITIVE', score=0.9, is_confident=True)
Health after:  {'model': 'distilbert-base-uncased-finetuned-sst-2-english', 'backend': 'stub', 'model_loaded': True}

**What this example tells us**
- One call (`predict`) runs the whole flow: clean → load → predict → format.
- The model loads lazily on the first call, as the health check shows.
- Each part can be replaced (for example a new `Preprocessor`) without changing the service.

**AI/ML Example:** Wrapping a sentiment model so a FastAPI endpoint needs only `service.predict(text)`.

**Business Example:** Marketing, support, and analytics teams all use the same tested prediction code.

**AI/ML Engineer Use Case:** Use the same service class in notebooks, batch jobs, and APIs.

## 9. Putting It All Together

**Concept:** Run the full service with batches, a stricter configuration, and failure cases.

**Why it's needed:** A service is only trustworthy if it behaves correctly for good input, bad input, and system failure.

In [9]:
texts = [
    "The delivery was great and the support was good",
    "Slow app, I hate the new update",
    "I love it",
    "The package arrived on Tuesday",
]

print("--- Batch prediction (threshold 0.60) ---")
for p in service.predict_batch(texts):
    print(f"{p.label:<9} {p.score:.2f} confident={p.is_confident} | {p.text}")

print("\n--- Stricter config (threshold 0.90) ---")
strict_service = InferenceService(replace(config, confidence_threshold=0.90))
preds = strict_service.predict_batch(texts)
print("Confident predictions:", sum(p.is_confident for p in preds), "of", len(preds))

print("\n--- Error handling ---")
for bad_input in ["", None]:
    try:
        service.predict(bad_input)
    except ModelServiceError as exc:
        print(f"{type(exc).__name__}: {exc}")

def failing_factory(cfg: ModelConfig) -> Backend:
    def model(batch: List[str]) -> List[Dict[str, Any]]:
        raise RuntimeError("out of memory")
    return model

broken = InferenceService(config, loader=ModelLoader(config, backend_factory=failing_factory))
try:
    broken.predict("Hello")
except ModelServiceError as exc:
    print(f"{type(exc).__name__}: {exc}")

--- Batch prediction (threshold 0.60) ---
INFO | model_service.loader | Model already loaded - reusing
INFO | model_service.inference | Running inference on 4 text(s)
WARNING | model_service.postprocessor | Low confidence (0.50) for text: The package arrived on Tuesday
POSITIVE  0.90 confident=True | The delivery was great and the support was good
NEGATIVE  0.90 confident=True | Slow app, I hate the new update
POSITIVE  0.70 confident=True | I love it
POSITIVE  0.50 confident=False | The package arrived on Tuesday

--- Stricter config (threshold 0.90) ---
INFO | model_service.loader | Loading model 'distilbert-base-uncased-finetuned-sst-2-english' (backend=stub)
INFO | model_service.loader | Model loaded successfully
INFO | model_service.inference | Running inference on 4 text(s)
WARNING | model_service.postprocessor | Low confidence (0.70) for text: I love it
WARNING | model_service.postprocessor | Low confidence (0.50) for text: The package arrived on Tuesday
Confident predictions: 2

**What this example tells us**
- Batch input works, and results always have the same structure.
- Changing only the config changes behaviour (fewer confident results at 0.90).
- Every failure - bad input or model failure - is a specific `ModelServiceError` that can be handled cleanly.

**AI/ML Example:** Low-confidence predictions (like the neutral sentence) are flagged for review instead of being trusted blindly.

**Business Example:** Automatic handling for clear cases, human review for uncertain ones - lower cost without losing quality.

**AI/ML Engineer Use Case:** Write unit tests using fake backends to check success, bad input, and failure paths without a real model.

## Summary – From Notebook Code to AI Engineering Code

| Principle | Where we used it |
|---|---|
| Separation of responsibilities | One class per job: Preprocessor, ModelLoader, PostProcessor, InferenceService |
| Configuration | `ModelConfig` (validated, immutable) |
| Error handling | `ModelServiceError` family |
| Logging | `get_logger` used in every component |
| Type hints & docstrings | Every class and method |
| Reusable functions | `predict`, `predict_batch`, `health` |
| Modular design | Parts replaceable through constructor arguments |

**Next step:** move each class into its own `.py` file, for example:

```
model_service/
├── config.py          # ModelConfig
├── logging_utils.py   # get_logger
├── errors.py          # ModelServiceError family
├── preprocessing.py   # Preprocessor
├── loader.py          # ModelLoader + backends
├── postprocessing.py  # Prediction, PostProcessor
└── service.py         # InferenceService
```

Then the notebook becomes a thin demo that only imports `InferenceService` and calls `predict()`.